# Lab 3: Gesture Recognition using Convolutional Neural Networks

In this lab you will train a convolutional neural network to make classifications on different hand gestures. By the end of the lab, you should be able to:

1. Load and split data for training, validation and testing
2. Train a Convolutional Neural Network
3. Apply transfer learning to improve your model

Note that for this lab we will not be providing you with any starter code. You should be able to take the code used in previous labs, tutorials and lectures and modify it accordingly to complete the tasks outlined below.

### What to submit

Submit a PDF file containing all your code, outputs, and write-up
from parts 1-5. You can produce a PDF of your Google Colab file by
going to **File > Print** and then save as PDF. The Colab instructions
has more information. Make sure to review the PDF submission to ensure that your answers are easy to read. Make sure that your text is not cut off at the margins. 

**Do not submit any other files produced by your code.**

Include a link to your colab file in your submission.

Please use Google Colab to complete this assignment. If you want to use Jupyter Notebook, please complete the assignment and upload your Jupyter Notebook file to Google Colab for submission. 

## Colab Link

Include a link to your colab file here

Colab Link: 

## Dataset

American Sign Language (ASL) is a complete, complex language that employs signs made by moving the
hands combined with facial expressions and postures of the body. It is the primary language of many
North Americans who are deaf and is one of several communication options used by people who are deaf or
hard-of-hearing. The hand gestures representing English alphabet are shown below. This lab focuses on classifying a subset
of these hand gesture images using convolutional neural networks. Specifically, given an image of a hand
showing one of the letters A-I, we want to detect which letter is being represented.

![alt text](https://www.disabled-world.com/pics/1/asl-alphabet.jpg)

## Part B. Building a CNN [50 pt]

For this lab, we are not going to give you any starter code. You will be writing a convolutional neural network
from scratch. You are welcome to use any code from previous labs, lectures and tutorials. You should also
write your own code.

You may use the PyTorch documentation freely. You might also find online tutorials helpful. However, all
code that you submit must be your own.

Make sure that your code is vectorized, and does not contain obvious inefficiencies (for example, unecessary
for loops, or unnecessary calls to unsqueeze()). Ensure enough comments are included in the code so that
your TA can understand what you are doing. It is your responsibility to show that you understand what you
write.

**This is much more challenging and time-consuming than the previous labs.** Make sure that you
give yourself plenty of time by starting early.

### 1. Data Loading and Splitting [5 pt]

Download the anonymized data provided on Quercus. To allow you to get a heads start on this project we will provide you with sample data from previous years. Split the data into training, validation, and test sets.

Note: Data splitting is not as trivial in this lab. We want our test set to closely resemble the setting in which
our model will be used. In particular, our test set should contain hands that are never seen in training!

Explain how you split the data, either by describing what you did, or by showing the code that you used.
Justify your choice of splitting strategy. How many training, validation, and test images do you have?

For loading the data, you can use plt.imread as in Lab 1, or any other method that you choose. You may find
torchvision.datasets.ImageFolder helpful. (see https://pytorch.org/docs/stable/torchvision/datasets.html?highlight=image%20folder#torchvision.datasets.ImageFolder
) 

In [3]:
import os
import random
from pathlib import Path
from collections import defaultdict
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

DATA_ROOT = Path(r"C:\Users\bernie\PycharmProjects\APS360_labs\Lab3\Lab3_Gestures_Summer")
random.seed(42)

# ── 1. Group images by person ─────────────────────────────────────────────────
all_images = list(DATA_ROOT.glob("**/*.jpg"))

person_gesture_groups = defaultdict(list)
for img_path in all_images:
    parts     = img_path.stem.split("_")
    global_id = int(parts[0])
    gesture   = parts[1]
    group_idx = (global_id - 1) // 3
    person_gesture_groups[(gesture, group_idx)].append(img_path)

person_ids = sorted(set(group_idx for (gesture, group_idx) in person_gesture_groups.keys()))
print(f"Total unique persons: {len(person_ids)}")

# ── 2. Split by person 70/15/15 ───────────────────────────────────────────────
random.shuffle(person_ids)
n       = len(person_ids)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_persons = set(person_ids[:n_train])
val_persons   = set(person_ids[n_train : n_train + n_val])
test_persons  = set(person_ids[n_train + n_val :])

train_imgs, val_imgs, test_imgs = [], [], []
for (gesture, group_idx), imgs in person_gesture_groups.items():
    if group_idx in train_persons:
        train_imgs.extend(imgs)
    elif group_idx in val_persons:
        val_imgs.extend(imgs)
    else:
        test_imgs.extend(imgs)

print(f"Train: {len(train_persons)} persons, {len(train_imgs)} images")
print(f"Val  : {len(val_persons)} persons,  {len(val_imgs)} images")
print(f"Test : {len(test_persons)} persons,  {len(test_imgs)} images")

# ── 3. Dataset class ──────────────────────────────────────────────────────────
class GestureDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform
        gestures         = sorted(set(p.stem.split("_")[1] for p in image_paths))
        self.class_to_idx = {g: i for i, g in enumerate(gestures)}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path  = self.image_paths[idx]
        img   = Image.open(path).convert("RGB")
        label = self.class_to_idx[path.stem.split("_")[1]]
        if self.transform:
            img = self.transform(img)
        return img, label

# ── 4. Transforms ─────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── 5. DataLoaders ────────────────────────────────────────────────────────────
train_dataset = GestureDataset(train_imgs, transform=train_transform)
val_dataset   = GestureDataset(val_imgs,   transform=eval_transform)
test_dataset  = GestureDataset(test_imgs,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0)

print(f"\nClasses: {train_dataset.class_to_idx}")
print(f"Train batches: {len(train_loader)}")

# after loading the data, and inspecting it by eyes, i decided to split the data first by parse the person id,because the structure is  after we shuffle it randomly, by strict 70 15 15 rule and build our set by then.

Total unique persons: 759
Train: 531 persons, 1553 images
Val  : 113 persons,  330 images
Test : 115 persons,  336 images

Classes: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8}
Train batches: 49


### 2. Model Building and Sanity Checking [15 pt]

### Part (a) Convolutional Network - 5 pt

Build a convolutional neural network model that takes the (224x224 RGB) image as input, and predicts the gesture
letter. Your model should be a subclass of nn.Module. Explain your choice of neural network architecture: how
many layers did you choose? What types of layers did you use? Were they fully-connected or convolutional?
What about other decisions like pooling layers, activation functions, number of channels / hidden units?

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GestureCNN(nn.Module):
    def __init__(self, num_classes=18):  # adjust num_classes to however many gestures you have
        super(GestureCNN, self).__init__()

        # ── Convolutional Block 1 ─────────────────────────────────────────────
        self.conv1 = nn.Conv2d(in_channels=3,  out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 224 → 112

        # ── Convolutional Block 2 ─────────────────────────────────────────────
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 112 → 56

        # ── Convolutional Block 3 ─────────────────────────────────────────────
        self.conv5 = nn.Conv2d(in_channels=64,  out_channels=128, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 56 → 28

        # ── Convolutional Block 4 ─────────────────────────────────────────────
        self.conv7 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28 → 14

        # ── Fully Connected Layers ────────────────────────────────────────────
        self.fc1 = nn.Linear(256 * 14 * 14, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)

        # ── Regularization ────────────────────────────────────────────────────
        self.dropout = nn.Dropout(p=0.5)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn1(self.conv2(x)))
        x = self.pool1(x)

        # Block 2
        x = F.relu(self.bn2(self.conv3(x)))
        x = F.relu(self.bn2(self.conv4(x)))
        x = self.pool2(x)

        # Block 3
        x = F.relu(self.bn3(self.conv5(x)))
        x = F.relu(self.bn3(self.conv6(x)))
        x = self.pool3(x)

        # Block 4
        x = F.relu(self.bn4(self.conv7(x)))
        x = F.relu(self.bn4(self.conv8(x)))
        x = self.pool4(x)

        # Flatten + FC
        x = x.view(x.size(0), -1)       # flatten: [batch, 256*14*14]
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)                  # raw logits, no softmax (CrossEntropyLoss handles it)

        return x

model = GestureCNN(num_classes=18)
dummy = torch.randn(4, 3, 224, 224)
out = model(dummy)
print(out.shape)

torch.Size([4, 18])


In [5]:
import torch
print(torch.cuda.is_available())    # True
print(torch.cuda.get_device_name(0)) # NVIDIA GeForce RTX 3070 Ti

True
NVIDIA GeForce RTX 3070 Ti


### Part (b) Training Code - 5 pt

Write code that trains your neural network given some training data. Your training code should make it easy
to tweak the usual hyperparameters, like batch size, learning rate, and the model object itself. Make sure
that you are checkpointing your models from time to time (the frequency is up to you). Explain your choice
of loss function and optimizer.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import time


# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-3
NUM_EPOCHS     = 30
WEIGHT_DECAY   = 1e-4
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Loss & Optimizer ──────────────────────────────────────────────────────────
model      = GestureCNN(num_classes=18).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ── Training Loop ─────────────────────────────────────────────────────────────
def train(model, train_loader, val_loader, num_epochs, criterion, optimizer, scheduler):
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(num_epochs):
        start = time.time()

        # ── Train phase ───────────────────────────────────────────────────────
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            correct       += predicted.eq(labels).sum().item()
            total         += labels.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validation phase ──────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs  = model(images)
                loss     = criterion(outputs, labels)

                val_loss_sum += loss.item() * images.size(0)
                _, predicted  = outputs.max(1)
                val_correct   += predicted.eq(labels).sum().item()
                val_total     += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total

        # ── Logging ───────────────────────────────────────────────────────────
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        elapsed = time.time() - start
        print(f"Epoch [{epoch+1:02d}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"Time: {elapsed:.1f}s")

        # ── Scheduler step ────────────────────────────────────────────────────
        scheduler.step()

        # ── Checkpointing every 5 epochs ──────────────────────────────────────
        if (epoch + 1) % 5 == 0:
            ckpt_path = CHECKPOINT_DIR / f"model_epoch{epoch+1}.pth"
            torch.save({
                'epoch':      epoch + 1,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss':   val_loss,
            }, ckpt_path)
            print(f"  ✓ Checkpoint saved → {ckpt_path}")

    return train_losses, val_losses, train_accs, val_accs


# ── Run ───────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_losses, val_losses, train_accs, val_accs = train(
    model, train_loader, val_loader,
    num_epochs=NUM_EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler
)

Using device: cuda
Using device: cuda
Epoch [01/30] Train Loss: 3.1740 Acc: 0.1082 | Val Loss: 2.5383 Acc: 0.1455 | Time: 6.3s
Epoch [02/30] Train Loss: 2.3936 Acc: 0.1069 | Val Loss: 2.3752 Acc: 0.0970 | Time: 5.8s
Epoch [03/30] Train Loss: 2.2453 Acc: 0.1449 | Val Loss: 2.3834 Acc: 0.1303 | Time: 5.8s
Epoch [04/30] Train Loss: 2.0268 Acc: 0.2048 | Val Loss: 2.3538 Acc: 0.1394 | Time: 5.8s
Epoch [05/30] Train Loss: 1.9000 Acc: 0.2543 | Val Loss: 2.5340 Acc: 0.1091 | Time: 5.8s
  ✓ Checkpoint saved → checkpoints\model_epoch5.pth
Epoch [06/30] Train Loss: 1.8416 Acc: 0.2814 | Val Loss: 2.5261 Acc: 0.1273 | Time: 5.8s
Epoch [07/30] Train Loss: 1.7378 Acc: 0.2910 | Val Loss: 2.3224 Acc: 0.0970 | Time: 5.9s
Epoch [08/30] Train Loss: 1.7197 Acc: 0.2904 | Val Loss: 2.3293 Acc: 0.1273 | Time: 5.8s
Epoch [09/30] Train Loss: 1.6958 Acc: 0.2968 | Val Loss: 2.2873 Acc: 0.1394 | Time: 5.8s
Epoch [10/30] Train Loss: 1.6651 Acc: 0.2898 | Val Loss: 2.2671 Acc: 0.1061 | Time: 5.8s
  ✓ Checkpoint saved

### Part (c) “Overfit” to a Small Dataset - 5 pt

One way to sanity check our neural network model and training code is to check whether the model is capable
of “overfitting” or “memorizing” a small dataset. A properly constructed CNN with correct training code
should be able to memorize the answers to a small number of images quickly.

Construct a small dataset (e.g. just the images that you have collected). Then show that your model and
training code is capable of memorizing the labels of this small data set.

With a large batch size (e.g. the entire small dataset) and learning rate that is not too high, You should be
able to obtain a 100% training accuracy on that small dataset relatively quickly (within 200 iterations).

### 3. Hyperparameter Search [15 pt]

### Part (a) - 3 pt

List 3 hyperparameters that you think are most worth tuning. Choose at least one hyperparameter related to
the model architecture.

### Part (b) - 5 pt

Tune the hyperparameters you listed in Part (a), trying as many values as you need to until you feel satisfied
that you are getting a good model. Plot the training curve of at least 4 different hyperparameter settings.

### Part (c) - 3 pt
Choose the best model out of all the ones that you have trained. Justify your choice.

### Part (d) - 4 pt
Report the test accuracy of your best model. You should only do this step once and prior to this step you should have only used the training and validation data.

### 4. Transfer Learning [15 pt]
For many image classification tasks, it is generally not a good idea to train a very large deep neural network
model from scratch due to the enormous compute requirements and lack of sufficient amounts of training
data.

One of the better options is to try using an existing model that performs a similar task to the one you need
to solve. This method of utilizing a pre-trained network for other similar tasks is broadly termed **Transfer
Learning**. In this assignment, we will use Transfer Learning to extract features from the hand gesture
images. Then, train a smaller network to use these features as input and classify the hand gestures.

As you have learned from the CNN lecture, convolution layers extract various features from the images which
get utilized by the fully connected layers for correct classification. AlexNet architecture played a pivotal
role in establishing Deep Neural Nets as a go-to tool for image classification problems and we will use an
ImageNet pre-trained AlexNet model to extract features in this assignment.

### Part (a) - 5 pt
Here is the code to load the AlexNet network, with pretrained weights. When you first run the code, PyTorch
will download the pretrained weights from the internet.

In [7]:
import torchvision.models
alexnet = torchvision.models.alexnet(pretrained=True)

C:\Users\bernie\PycharmProjects\APS360_labs\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\bernie\PycharmProjects\APS360_labs\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\bernie/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth


100.0%


The alexnet model is split up into two components: *alexnet.features* and *alexnet.classifier*. The
first neural network component, *alexnet.features*, is used to compute convolutional features, which are
taken as input in *alexnet.classifier*.

The neural network alexnet.features expects an image tensor of shape Nx3x224x224 as input and it will
output a tensor of shape Nx256x6x6 . (N = batch size).

Compute the AlexNet features for each of your training, validation, and test data. Here is an example code
snippet showing how you can compute the AlexNet features for some images (your actual code might be
different):

In [ ]:
# img = ... a PyTorch tensor with shape [N,3,224,224] containing hand images ...
features = alexnet.features(img)

**Save the computed features**. You will be using these features as input to your neural network in Part
(b), and you do not want to re-compute the features every time. Instead, run *alexnet.features* once for
each image, and save the result.

### Part (b) - 3 pt
Build a convolutional neural network model that takes as input these AlexNet features, and makes a
prediction. Your model should be a subclass of nn.Module.

Explain your choice of neural network architecture: how many layers did you choose? What types of layers
did you use: fully-connected or convolutional? What about other decisions like pooling layers, activation
functions, number of channels / hidden units in each layer?

Here is an example of how your model may be called:

In [ ]:
# features = ... load precomputed alexnet.features(img) ...
output = model(features)
prob = F.softmax(output)

### Part (c) - 5 pt
Train your new network, including any hyperparameter tuning. Plot and submit the training curve of your
best model only.

Note: Depending on how you are caching (saving) your AlexNet features, PyTorch might still be tracking
updates to the **AlexNet weights**, which we are not tuning. One workaround is to convert your AlexNet
feature tensor into a numpy array, and then back into a PyTorch tensor.

In [ ]:
tensor = torch.from_numpy(tensor.detach().numpy())

### Part (d) - 2 pt
Report the test accuracy of your best model. How does the test accuracy compare to Part 3(d) without transfer learning?